# CWA: Curie-Weiss Activation
## ResNet-20 CIFAR-100 Width Analysis + Overhead Benchmark

This notebook demonstrates the **Curie-Weiss Activation (CWA)**, a novel activation function derived from statistical physics mean-field theory.

**Core idea:** CWA replaces a standard pointwise activation with a *self-consistent* equation:

$$y_i = \tanh\!\left(x_i + J \cdot \overline{y}\right), \quad \overline{y} = \frac{1}{n}\sum_j y_j$$

where $J$ is a **per-layer learnable coupling strength** (analogous to the Curie-Weiss coupling in the Ising model).

**Backprop strategy (hybrid):**
- `J·s̄ < 0.8` (sub-critical): **unrolled autograd** through K_TRAIN_UNROLLED fixed steps
- `J·s̄ ≥ 0.8` (near-critical): **Implicit Function Theorem (IFT) backward** — O(1) activation memory

**Experiments:**
- **Exp 2**: ResNet-20 on CIFAR-100, four configs (standard/wide × BN/no-BN), CWA vs GELU/SELU/tanhLN/GELULN
- **Exp 5**: Synthetic overhead benchmark measuring wall-clock and memory vs GELU across J·s̄ targets

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — not pre-installed on Colab
_pip('loguru==0.7.2')

# Core scientific packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cpu')
    _pip('numpy==2.0.2', 'matplotlib==3.10.0', 'scipy==1.16.3')

In [ ]:
import gc
import json
import math
import os
import sys
import time

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

print(f'torch={torch.__version__}, device={"cuda" if torch.cuda.is_available() else "cpu"}')

# ── CWA implementation (from cwa.py) ─────────────────────────────────────────

K_TRAIN_UNROLLED = 8   # fixed steps in unrolled mode (J*s_bar < 0.8)
K_TRAIN_IFT = 20       # fixed steps in IFT mode (near-critical)


class CWAFunction(torch.autograd.Function):
    """IFT-based custom backward. Used when J*s_bar >= 0.8."""

    @staticmethod
    def forward(ctx, x, J_raw, k_iters: int = 20):
        J = torch.sigmoid(J_raw)
        n = x.shape[1]

        m = torch.zeros(x.shape[0], 1, *x.shape[2:], device=x.device, dtype=x.dtype)
        for _ in range(k_iters):
            m = torch.tanh(x + J * m).mean(dim=1, keepdim=True)

        h_star = x + J * m
        sech2 = 1.0 / torch.cosh(h_star) ** 2
        s_bar = sech2.mean()
        J_s_bar = J * s_bar
        y = torch.tanh(h_star)

        ctx.save_for_backward(x, m, J_raw, sech2, s_bar, J_s_bar)
        ctx.n = n
        ctx.k_iters = k_iters
        ctx.J_s_bar_val = J_s_bar.item()

        return y, J_s_bar.detach(), torch.tensor(float(k_iters), device=x.device)

    @staticmethod
    def backward(ctx, grad_y, _g1, _g2):
        x, m_star, J_raw, sech2, s_bar, J_s_bar = ctx.saved_tensors
        J = torch.sigmoid(J_raw)
        n = ctx.n

        denom = (1.0 - J_s_bar).clamp(min=1e-3)
        G = (grad_y * sech2).sum(dim=1, keepdim=True)
        grad_x = sech2 * (grad_y + J * G / (n * denom))

        grad_J = (grad_y * sech2 * m_star / denom).sum()
        grad_J_raw = grad_J * J * (1.0 - J)

        return grad_x, grad_J_raw, None


class CWA(nn.Module):
    """
    Curie-Weiss Activation: y_i = tanh(x_i + J * mean_channels(y))
    J = sigmoid(J_raw) in (0,1) is a per-layer learnable scalar.

    Backprop modes:
      - unrolled: K_TRAIN_UNROLLED fixed steps, autograd tracks through them.
      - IFT: K_TRAIN_IFT fixed steps, IFT backward (no autograd through iterations).

    Mode decision uses cached J*s_bar from previous forward pass — avoids an extra
    no_grad probe on every batch (which was ~40% of CWA wall-clock).
    """

    def __init__(self, K_max: int = 50):
        super().__init__()
        self.J_raw = nn.Parameter(torch.zeros(1))
        self.K_max = K_max  # used only in benchmark mode
        self.last_J = None
        self.last_J_s_bar = None
        self.last_k = None
        self.last_mode = None
        self._prev_use_ift = False
        self.benchmark_mode = False  # if True, use K_max with convergence check

    def _forward_train(self, x, J):
        """Fast training forward: fixed K steps, no per-step convergence check."""
        if self._prev_use_ift:
            y, J_s_bar_t, k_t = CWAFunction.apply(x, self.J_raw, K_TRAIN_IFT)
            mode = 'IFT'
        else:
            # Unrolled: K_TRAIN_UNROLLED steps through autograd
            m = torch.zeros(x.shape[0], 1, *x.shape[2:], device=x.device, dtype=x.dtype)
            for _ in range(K_TRAIN_UNROLLED):
                m = torch.tanh(x + J * m).mean(dim=1, keepdim=True)
            y = torch.tanh(x + J * m)
            with torch.no_grad():
                sech2_f = 1.0 / torch.cosh(x + J * m.detach()) ** 2
                J_s_bar_t = J * sech2_f.mean()
            k_t = torch.tensor(float(K_TRAIN_UNROLLED))
            mode = 'unrolled'
        return y, J_s_bar_t, k_t, mode

    def _forward_benchmark(self, x, J):
        """Benchmark forward: full convergence checking with K_max iterations."""
        m = torch.zeros(x.shape[0], 1, *x.shape[2:], device=x.device, dtype=x.dtype)
        k_final = 0
        for k in range(self.K_max):
            h = x + J * m
            m_new = torch.tanh(h).mean(dim=1, keepdim=True)
            with torch.no_grad():
                sech2_tmp = 1.0 / torch.cosh(h) ** 2
                J_s_bar_tmp = J * sech2_tmp.mean()
                delta = 1e-4 * (1.0 - J_s_bar_tmp.clamp(max=0.9999))
                diff = (m_new - m).abs().max()
            m = m_new
            k_final = k + 1
            if diff.item() < delta.item():
                break

        use_ift = self._prev_use_ift
        if use_ift:
            y, J_s_bar_t, k_t = CWAFunction.apply(x, self.J_raw, k_final)
            mode = 'IFT'
        else:
            y = torch.tanh(x + J * m)
            with torch.no_grad():
                sech2_f = 1.0 / torch.cosh(x + J * m.detach()) ** 2
                J_s_bar_t = J * sech2_f.mean()
            k_t = torch.tensor(float(k_final))
            mode = 'unrolled'
        return y, J_s_bar_t, k_t, mode

    def forward(self, x):
        J = torch.sigmoid(self.J_raw)

        if self.benchmark_mode:
            y, J_s_bar_t, k_t, mode = self._forward_benchmark(x, J)
        else:
            y, J_s_bar_t, k_t, mode = self._forward_train(x, J)

        J_s_bar_val = J_s_bar_t.item() if torch.is_tensor(J_s_bar_t) else float(J_s_bar_t)
        self._prev_use_ift = J_s_bar_val >= 0.8

        self.last_J = J.item()
        self.last_J_s_bar = J_s_bar_val
        self.last_k = k_t.item() if torch.is_tensor(k_t) else float(k_t)
        self.last_mode = mode
        return y

    def reset_cache(self):
        self._prev_use_ift = False

print('CWA class defined.')

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2f6f35-curie-weiss-activation-formal-verificati/main/round-1/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f: return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

In [ ]:
data = load_data()
n_examples = len(data['datasets'][0]['examples'])
print(f'Loaded {n_examples} examples')
verdict_preview = json.dumps(data["metadata"]["verdict"], indent=2)
print(f'Verdict:\n{verdict_preview[:500]}')

## Configuration

All tunable parameters are set here. Start with minimal values for a fast demo run.
The original experiment used 10 epochs, 4 configs, multiple activations, and 8 overhead targets.

In [ ]:
# ── Tunable parameters ────────────────────────────────────────────────────────
# CWA unit demo
DEMO_BATCH = 4           # batch size for synthetic CWA demo (orig: 32)
DEMO_CHANNELS = 8        # channels for synthetic CWA demo (orig: 256)
DEMO_J_VALUES = [0.3, 0.7, 0.95]   # J values to sweep (orig: 8 targets)

# Overhead benchmark (synthetic only, no CIFAR)
BENCH_N_TIMING = 5       # timing repetitions per target (orig: 20)
BENCH_N_WARMUP = 2       # warmup repetitions (orig: 5)
BENCH_BATCH = 4          # batch size (orig: 32)
BENCH_C = 32             # channels (orig: 256)
BENCH_H, BENCH_W = 4, 4 # spatial (orig: 8x8)

# Training configs to display from loaded data (subset for readability)
CONFIGS_TO_SHOW = ['standard_no_bn', 'standard_bn', 'wide_no_bn']  # all 4 available

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## CWA Unit Tests and Synthetic Demo

Verify the CWA implementation works correctly:
1. Shape preservation and tanh-range constraint
2. Gradient flow through J_raw (learnable coupling)
3. Mode switching: unrolled (sub-critical) → IFT (near-critical) at J·s̄ = 0.8
4. Sweep J values to show how coupling strength affects the mean-field order parameter

In [ ]:
# ── Unit test: shape, range, gradients ───────────────────────────────────────
x_test = torch.randn(1, DEMO_CHANNELS)
cwa_test = CWA()
y_test = cwa_test(x_test)
assert y_test.shape == x_test.shape, 'Shape mismatch'
assert not torch.isnan(y_test).any(), 'NaN in output'
assert (y_test.abs() <= 1.0 + 1e-4).all(), 'tanh range violated'
print(f'[PASS] Shape/range: J={cwa_test.last_J:.3f}, J*s_bar={cwa_test.last_J_s_bar:.3f}, k={cwa_test.last_k}, mode={cwa_test.last_mode}')

# Gradient test
x_g = torch.randn(DEMO_BATCH, DEMO_CHANNELS, requires_grad=True)
cwa_g = CWA()
y_g = cwa_g(x_g)
y_g.sum().backward()
assert x_g.grad is not None and not torch.isnan(x_g.grad).any(), 'Bad x.grad'
assert cwa_g.J_raw.grad is not None and not torch.isnan(cwa_g.J_raw.grad).any(), 'Bad J_raw.grad'
print(f'[PASS] Gradient flow: J_raw.grad={cwa_g.J_raw.grad.item():.4f}')

# IFT mode test (high J → near-critical)
cwa_hi = CWA()
with torch.no_grad():
    cwa_hi.J_raw.fill_(4.0)  # sigmoid(4.0) ≈ 0.982 → high J → J*s_bar >= 0.8
x_hi = torch.randn(DEMO_BATCH, DEMO_CHANNELS) * 0.01
_ = cwa_hi(x_hi)   # first call warms up cache
y_hi = cwa_hi(x_hi)  # second call uses correct cached mode
assert cwa_hi.last_mode == 'IFT', f'Expected IFT, got {cwa_hi.last_mode}'
print(f'[PASS] IFT mode: J*s_bar={cwa_hi.last_J_s_bar:.3f}')

# Unrolled mode test (low J → sub-critical)
cwa_lo = CWA()
with torch.no_grad():
    cwa_lo.J_raw.fill_(-2.0)  # sigmoid(-2.0) ≈ 0.119 → low J
y_lo = cwa_lo(torch.randn(DEMO_BATCH, DEMO_CHANNELS) * 0.01)
assert cwa_lo.last_mode == 'unrolled', f'Expected unrolled, got {cwa_lo.last_mode}'
print(f'[PASS] Unrolled mode: J*s_bar={cwa_lo.last_J_s_bar:.3f}')

print('\nAll unit tests PASSED!')

# ── Sweep J values ────────────────────────────────────────────────────────────
print('\nJ sweep (mean-field order parameter):')
print(f'{"J_raw":>8} {"J":>8} {"J*s_bar":>10} {"mode":>10} {"k":>5}')
for j_val in DEMO_J_VALUES:
    cwa_sweep = CWA()
    with torch.no_grad():
        # Set J_raw = logit(j_val)
        cwa_sweep.J_raw.fill_(math.log(j_val / (1 - j_val)))
    x_sw = torch.randn(DEMO_BATCH, DEMO_CHANNELS) * 0.01
    _ = cwa_sweep(x_sw)   # warm-up cache
    _ = cwa_sweep(x_sw)   # second pass with correct mode
    print(f'{cwa_sweep.J_raw.item():>8.3f} {cwa_sweep.last_J:>8.3f} {cwa_sweep.last_J_s_bar:>10.4f} {cwa_sweep.last_mode:>10} {cwa_sweep.last_k:>5.0f}')

## Synthetic Overhead Benchmark (mini-scale)

Measure wall-clock time and memory ratio (CWA vs GELU) across J·s̄ targets.
This is a mini-scale version — the full benchmark used batch=32, C=256, H=W=8.

Key prediction: overhead grows with J·s̄ because more fixed-point iterations K* are needed near criticality. The IFT backward kicks in at J·s̄ ≥ 0.8.

In [ ]:
def logit(t):
    t = max(min(t, 0.99), 0.01)
    return torch.tensor([math.log(t / (1 - t))], dtype=torch.float32)


def measure_cwa_overhead_mini(device, targets, n_warmup=BENCH_N_WARMUP, n_timing=BENCH_N_TIMING):
    overhead_table = []
    gelu = nn.GELU().to(device)

    for target in targets:
        cwa = CWA(K_max=100).to(device)
        cwa.benchmark_mode = True
        with torch.no_grad():
            cwa.J_raw.copy_(logit(target).to(device))
        cwa.J_raw.requires_grad_(False)

        x = torch.randn(BENCH_BATCH, BENCH_C, BENCH_H, BENCH_W, device=device) * 0.01
        x.requires_grad_(True)

        # Warmup CWA
        for _ in range(n_warmup):
            y = cwa(x); y.sum().backward()
            if x.grad is not None: x.grad.zero_()

        # Time CWA
        times_cwa = []
        for _ in range(n_timing):
            t0 = time.perf_counter()
            y = cwa(x); y.sum().backward()
            times_cwa.append(time.perf_counter() - t0)
            if x.grad is not None: x.grad.zero_()

        actual_J_s_bar = cwa.last_J_s_bar
        actual_k = cwa.last_k
        actual_mode = cwa.last_mode

        # Warmup GELU
        x_gelu = x.detach().requires_grad_(True)
        for _ in range(n_warmup):
            y_g = gelu(x_gelu); y_g.sum().backward()
            if x_gelu.grad is not None: x_gelu.grad.zero_()

        # Time GELU
        times_gelu = []
        for _ in range(n_timing):
            t0 = time.perf_counter()
            y_g = gelu(x_gelu); y_g.sum().backward()
            times_gelu.append(time.perf_counter() - t0)
            if x_gelu.grad is not None: x_gelu.grad.zero_()

        wall_cwa = sum(times_cwa) / len(times_cwa) * 1000
        wall_gelu = sum(times_gelu) / len(times_gelu) * 1000

        row = {
            'J_s_bar_target': target,
            'J_s_bar_actual': actual_J_s_bar,
            'K_star': actual_k,
            'backprop_mode': actual_mode,
            'wall_clock_ms_cwa': wall_cwa,
            'wall_clock_ms_gelu': wall_gelu,
            'wall_clock_ratio': wall_cwa / max(wall_gelu, 1e-9),
        }
        overhead_table.append(row)
        print(f'  target={target:.2f}: K*={actual_k:.0f}, mode={actual_mode:>8}, '
              f'wall_ratio={row["wall_clock_ratio"]:.1f}x')

    return overhead_table


print('Running mini overhead benchmark...')
bench_targets = [0.3, 0.7, 0.9]  # subset of DEMO_J_VALUES for speed
overhead_mini = measure_cwa_overhead_mini(DEVICE, bench_targets)
print('Done.')

## Results: Training Curves and Overhead

Load and visualize results from the pre-computed CIFAR-100 experiment stored in `mini_demo_data.json`:
- Training accuracy curves per epoch for each config/activation
- J·s̄ (coupling strength) evolution across ResNet blocks
- Overhead ratio (wall-clock) vs J·s̄ target from the full benchmark

In [ ]:
# ── Extract structured results from loaded data ───────────────────────────────
resnet_results = data['metadata']['resnet20_results_summary']
overhead_table_full = data['metadata']['overhead_table']
verdict = data['metadata']['verdict']
width_correlation = data['metadata']['width_correlation']

print('=== CIFAR-100 Final Accuracies ===')
print(f'{"Config":<20} {"Activation":<12} {"Acc Mean":>10} {"Acc Std":>10}')
print('-' * 56)
for cfg_label in CONFIGS_TO_SHOW:
    cfg = resnet_results.get(cfg_label, {})
    for act_name, agg in cfg.items():
        print(f'{cfg_label:<20} {act_name:<12} {agg["test_acc_mean"]:>10.4f} {agg["test_acc_std"]:>10.4f}')
    print()

print('=== Verdict ===')
for k, v in verdict.items():
    print(f'  {k}: {v}')

## Visualization

Four panels:
1. Training accuracy curves (standard_no_bn) — CWA vs all baselines
2. Training accuracy curves (standard_bn) — CWA vs GELU
3. Per-block J·s̄ at final epoch (standard_no_bn) — mean-field coupling landscape
4. Overhead ratio vs J·s̄ target from the full GPU benchmark

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CWA (Curie-Weiss Activation) — CIFAR-100 ResNet-20 Results', fontsize=14, fontweight='bold')

# ── Panel 1: standard_no_bn training curves ──────────────────────────────────
ax = axes[0, 0]
cfg = resnet_results.get('standard_no_bn', {})
colors = {'CWA': 'tab:blue', 'GELU': 'tab:orange', 'SELU': 'tab:green', 'tanhLN': 'tab:red', 'GELULN': 'tab:purple'}
for act_name, agg in cfg.items():
    for seed_hist in agg['acc_history_per_seed']:
        epochs = list(range(len(seed_hist)))
        ax.plot(epochs, [a * 100 for a in seed_hist],
                color=colors.get(act_name, 'gray'),
                linewidth=2.0 if act_name == 'CWA' else 1.2,
                linestyle='-' if act_name == 'CWA' else '--',
                label=f'{act_name} ({agg["test_acc_mean"]*100:.1f}%)')
ax.set_title('standard_no_bn (widths 16/32/64, no BN)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy (%)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Panel 2: standard_bn training curves ─────────────────────────────────────
ax = axes[0, 1]
cfg = resnet_results.get('standard_bn', {})
for act_name, agg in cfg.items():
    for seed_hist in agg['acc_history_per_seed']:
        epochs = list(range(len(seed_hist)))
        ax.plot(epochs, [a * 100 for a in seed_hist],
                color=colors.get(act_name, 'gray'),
                linewidth=2.0 if act_name == 'CWA' else 1.2,
                linestyle='-' if act_name == 'CWA' else '--',
                label=f'{act_name} ({agg["test_acc_mean"]*100:.1f}%)')
ax.set_title('standard_bn (widths 16/32/64, with BN)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy (%)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Panel 3: per-block J*s_bar at final epoch (standard_no_bn CWA) ───────────
ax = axes[1, 0]
cwa_data = resnet_results.get('standard_no_bn', {}).get('CWA', {})
final_js = cwa_data.get('final_J_s_bar_per_block', {})

group_colors = {}
for bname, vals in final_js.items():
    if 'group1' in bname: group_colors[bname] = 'tab:blue'
    elif 'group2' in bname: group_colors[bname] = 'tab:orange'
    elif 'group3' in bname: group_colors[bname] = 'tab:green'
    elif 'act0' in bname: group_colors[bname] = 'tab:red'
    else: group_colors[bname] = 'gray'

block_names = list(final_js.keys())
y_vals = [float(final_js[b][0]) if final_js[b] else 0 for b in block_names]
bar_colors = [group_colors.get(b, 'gray') for b in block_names]
ax.bar(range(len(block_names)), y_vals, color=bar_colors, alpha=0.8)
ax.axhline(0.8, color='red', linestyle='--', linewidth=1.5, label='IFT threshold (0.8)')
ax.axhline(0.7, color='orange', linestyle=':', linewidth=1.5, label='SOC threshold (0.7)')
ax.set_xticks(range(len(block_names)))
ax.set_xticklabels([b.replace('group', 'g') for b in block_names], rotation=60, ha='right', fontsize=7)
ax.set_title('Final J·s̄ per block (standard_no_bn, CWA)')
ax.set_ylabel('J·s̄'); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='tab:blue', label='group1 (16ch)'),
                   Patch(facecolor='tab:orange', label='group2 (32ch)'),
                   Patch(facecolor='tab:green', label='group3 (64ch)'),
                   Patch(facecolor='tab:red', label='stem (act0)')]
ax.legend(handles=legend_elements, fontsize=7, loc='upper right')

# ── Panel 4: overhead ratio from full GPU benchmark ───────────────────────────
ax = axes[1, 1]
targets = [r['J_s_bar_target'] for r in overhead_table_full]
wall_ratios = [r['wall_clock_ratio'] for r in overhead_table_full]
mem_ratios = [r['memory_ratio'] for r in overhead_table_full]
modes = [r['backprop_mode'] for r in overhead_table_full]

ax.plot(targets, wall_ratios, 'o-', color='tab:blue', linewidth=2, markersize=7, label='Wall-clock ratio (CWA/GELU)')
ax.plot(targets, mem_ratios, 's--', color='tab:orange', linewidth=2, markersize=7, label='Memory ratio (CWA/GELU)')
ax.axhline(2.0, color='red', linestyle=':', linewidth=1.5, label='2× overhead threshold')
ax.axvline(0.8, color='gray', linestyle='--', linewidth=1, label='IFT switch (J·s̄=0.8)')

# Mark IFT points
for i, (t, wr, mode) in enumerate(zip(targets, wall_ratios, modes)):
    if mode == 'IFT':
        ax.annotate('IFT', (t, wr), textcoords='offset points', xytext=(5, 5), fontsize=7, color='tab:blue')

ax.set_title('CWA vs GELU Overhead (full GPU benchmark)')
ax.set_xlabel('J·s̄ target'); ax.set_ylabel('Ratio (CWA / GELU)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('cwa_results.png', dpi=100, bbox_inches='tight')
plt.show()
print('Figure saved to cwa_results.png')

## Summary

The table below summarizes the four success criteria evaluated by the experiment:

In [ ]:
# ── Print final verdict table ─────────────────────────────────────────────────
print('=' * 65)
print('CWA EXPERIMENT VERDICT')
print('=' * 65)

criteria = [
    ('Memory overhead ≤ 2x', verdict.get('memory_within_2x')),
    ('Width → J·s̄ positive correlation', verdict.get('width_positive_correlation')),
    ('CWA acc gain > 0.5% over GELU (no-BN)', verdict.get('cwa_vs_gelu_no_bn_significant')),
    ('Self-organized criticality (mean J·s̄ > 0.7)', verdict.get('soc_observed')),
]

for criterion, passed in criteria:
    status = 'PASS' if passed else ('FAIL' if passed is False else 'N/A')
    icon = '✓' if passed else ('✗' if passed is False else '?')
    print(f'  {icon} [{status:4s}] {criterion}')

print()
cwa_acc = verdict.get('cwa_acc_standard_no_bn', 0)
gelu_acc = verdict.get('gelu_acc_standard_no_bn', 0)
print(f'CWA accuracy  (standard_no_bn): {cwa_acc:.4f}')
print(f'GELU accuracy (standard_no_bn): {gelu_acc:.4f}')
print(f'Gain: {cwa_acc - gelu_acc:+.4f}')
print(f'Mean final J·s̄: {verdict.get("mean_final_J_s_bar", 0):.4f}')

# Mini overhead benchmark results
print()
print('Mini overhead benchmark (this notebook):')
print(f'{"Target":>8} {"K*":>5} {"Mode":>10} {"Wall ratio":>12}')
for row in overhead_mini:
    print(f'{row["J_s_bar_target"]:>8.2f} {row["K_star"]:>5.0f} {row["backprop_mode"]:>10} {row["wall_clock_ratio"]:>12.1f}x')